In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

RAW_DIR = "sources/catastro"
OUT_DIR = "data"

# Cadastral references (parcel level, 14 characters):
RC_COLEGIATA = "0139402VK4703G"   # Calle de Toledo 37
RC_INSTITUTO = "0139424VK4703G"   # Calle de Toledo 39
MANZANA = "01394"                 # First 5 characters shared by all parcels of the block

# Search box in EPSG:25830 (xmin, ymin, xmax, ymax), about 300 x 300 m around the block:
BBOX = (439850, 4473650, 440150, 4473950)

WFS_CP = "http://ovc.catastro.meh.es/INSPIRE/wfsCP.aspx"
WFS_BU = "http://ovc.catastro.meh.es/INSPIRE/wfsBU.aspx"

In [ ]:
# Celda 2
import os
import requests
import geopandas as gpd

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Celda 3
# Download all cadastral parcels inside the search box:
params = {
    "service": "wfs",
    "version": "2",
    "request": "getfeature",
    "typeNames": "CP:CadastralParcel",
    "srsname": "EPSG::25830",
    "bbox": ",".join(str(v) for v in BBOX),
}
answer = requests.get(WFS_CP, params=params, timeout=120)
parcels_file = RAW_DIR + "/cp_bbox.gml"
with open(parcels_file, "wb") as f:
    f.write(answer.content)
print(answer.status_code, len(answer.content), "bytes ->", parcels_file)

In [ ]:
# Celda 4
# Read the parcels and keep only the ones of our block:
parcels = gpd.read_file(parcels_file)
parcels = parcels.set_crs(25830, allow_override=True)
print(len(parcels), "parcels in the box")

block = parcels[parcels["nationalCadastralReference"].str.startswith(MANZANA)].copy()
block["area_m2"] = block.geometry.area.round(1)
print(len(block), "parcels in block", MANZANA)
block[["nationalCadastralReference", "area_m2"]].sort_values("nationalCadastralReference")

In [ ]:
# Celda 5
# Ask the Catastro the address and use of each parcel of the block:
# "0139402VK4703G" -> pc1 = "0139402", pc2 = "VK4703G"
URL_DNP = "http://ovc.catastro.meh.es/ovcservweb/OVCSWLocalizacionRC/OVCCallejero.asmx/Consulta_DNPRC"

addresses = {}
uses = {}
for rc in block["nationalCadastralReference"]:
    params = {"Provincia": "", "Municipio": "", "RC": rc}
    text = requests.get(URL_DNP, params=params, timeout=60).text
    address = ""
    use = ""
    if "<ldt>" in text:
        address = text.split("<ldt>")[1].split("</ldt>")[0]
    if "<luso>" in text:
        use = text.split("<luso>")[1].split("</luso>")[0]
    addresses[rc] = address
    uses[rc] = use

block["address"] = block["nationalCadastralReference"].map(addresses)
block["use"] = block["nationalCadastralReference"].map(uses)
block[["nationalCadastralReference", "area_m2", "use", "address"]].sort_values("nationalCadastralReference")

In [ ]:
# Celda 6
# Download the building parts (footprints with number of floors) of every parcel of the block:
parts_list = []
for rc in block["nationalCadastralReference"]:
    params = {
        "service": "wfs",
        "version": "2",
        "request": "getfeature",
        "STOREDQUERIE_ID": "GETBUILDINGPARTBYPARCEL",
        "refcat": rc,
        "srsname": "EPSG::25830",
    }
    answer = requests.get(WFS_BU, params=params, timeout=120)
    parts_file = RAW_DIR + "/bu_parts_" + rc + ".gml"
    with open(parts_file, "wb") as f:
        f.write(answer.content)
    parts = gpd.read_file(parts_file)
    parts["parcel"] = rc
    parts_list.append(parts)
    print(rc, len(parts), "building parts")

In [ ]:
# Celda 7
# Join all building parts in one table:
import pandas as pd

parts = gpd.GeoDataFrame(pd.concat(parts_list, ignore_index=True), crs=25830)
parts = parts.set_crs(25830, allow_override=True)
parts["area_m2"] = parts.geometry.area.round(1)
print(len(parts), "building parts in the block")
print(list(parts.columns))

In [ ]:
# Celda 8
# Summary per parcel: floors above ground of each part:
columns = ["parcel", "numberOfFloorsAboveGround", "numberOfFloorsBelowGround", "area_m2"]
parts[columns].sort_values(["parcel", "numberOfFloorsAboveGround"])

In [ ]:
# Celda 9
# Save everything in one GeoPackage:
gpkg = OUT_DIR + "/catastro_manzana.gpkg"
block.to_file(gpkg, layer="parcelas", driver="GPKG")
parts.to_file(gpkg, layer="building_parts", driver="GPKG")
print("Saved:", gpkg)